# Project 4 · Listening Past the Noise: pretrained audio embeddings with a noise-aware classifier

This notebook builds a complete submission in four steps.

1. **Three views of each recording.** AST (an AudioSet-pretrained Audio Spectrogram Transformer, the strongest view), PANNs CNN14 embeddings (a second pretrained model), and the librosa statistics your teammates used.
2. **Noise-aware training.** A logistic-regression model per view is trained on the 620 verified recordings, plus the unverified recordings that a verified-only model agrees with (or weights by how plausible their label looks). Cross-validation on the verified recordings chooses the setting.
3. **Calibration and class priors.** The views are averaged, sharpened by a fitted temperature and corrected towards the class sizes in `train.csv`.
4. **Output.** `submission.csv` gives three ranked categories for every test recording.

**Before you run**
- **Check the rules first.** Pretrained models must be allowed by your course. If they are not, set `USE_AST` and `USE_PANNS` to `False`: the notebook then runs on the librosa view alone.
- **On Kaggle:** Settings → Accelerator → GPU, and Settings → Internet → On (the pretrained weights download once). The competition data is found automatically under `/kaggle/input`.
- **On your own computer:** keep this notebook next to the `project4` folder, or set the environment variable `P4_DATA` to the folder that contains `train.csv`. AST needs an NVIDIA GPU; without one the notebook skips it and says so.
- Feature extraction is cached in the working folder, so a second run only repeats the modelling.

Expected time on a Kaggle GPU: about 30–40 minutes for the first run.

**What to expect.** The target public MAP@3 is about 0.94. That is what the design aims for; it has not been measured yet. Read the cross-validation table in section 3 before using one of your daily submissions.

In [ ]:
# 0. Setup: switches, locations, data checks -------------------------------
import os, sys, time, warnings, importlib.util, contextlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

# ---- switches -------------------------------------------------------------
USE_AST    = True    # AudioSet-pretrained Audio Spectrogram Transformer: the strongest view (needs a GPU)
USE_PANNS  = True    # PANNs CNN14 embeddings: a second, different pretrained view
USE_HAND   = True    # librosa statistics, the features your teammates used (CPU only)
AST_ON_CPU = False   # AST on a CPU takes many hours; keep False unless you really want to wait
C_LR       = 0.1     # logistic-regression regularisation strength (smaller = stronger)
GAMMA      = 0.5     # prior correction towards training class sizes: 0 = off, 1 = full
N_FOLDS    = 5
CV_SEED    = 42

# ---- where things live ----------------------------------------------------
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("P4_WORK", Path.cwd() / "p4_work"))
WORK.mkdir(parents=True, exist_ok=True)

def find_data_dir():
    """The folder that holds train.csv, test.csv and sample_submission.csv."""
    if os.environ.get("P4_DATA"):
        return Path(os.environ["P4_DATA"])
    for root in [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent, Path.home() / "Downloads"]:
        if not root.exists():
            continue
        for tr in root.rglob("train.csv"):
            d = tr.parent
            if (d / "test.csv").exists() and (d / "sample_submission.csv").exists():
                return d
    raise FileNotFoundError("Could not find train.csv, test.csv and sample_submission.csv. "
                            "Set the environment variable P4_DATA to the folder that contains them.")

def find_audio_dir(split):
    """audio/<split> or audio/audio/<split> (the local copy uses the second layout)."""
    for cand in [DATA / "audio" / split, DATA / "audio" / "audio" / split]:
        if cand.is_dir() and any(cand.glob("*.wav")):
            return cand
    for cand in DATA.rglob(split):
        if cand.is_dir() and any(cand.glob("*.wav")):
            return cand
    raise FileNotFoundError(f"no folder of .wav files named '{split}' under {DATA}")

DATA = find_data_dir()
TRAIN_AUDIO, TEST_AUDIO = find_audio_dir("train"), find_audio_dir("test")
train_df  = pd.read_csv(DATA / "train.csv", dtype={"id": str, "label": str})
test_df   = pd.read_csv(DATA / "test.csv", dtype={"id": str})
sample_df = pd.read_csv(DATA / "sample_submission.csv", dtype={"id": str})
train_ids, test_ids = train_df["id"].tolist(), test_df["id"].tolist()

assert train_df["id"].is_unique and test_df["id"].is_unique
assert set(test_ids) == set(sample_df["id"]) and len(sample_df) == len(test_ids), \
    "test.csv and sample_submission.csv do not list the same recordings"
missing = [i for i in train_ids if not (TRAIN_AUDIO / f"{i}.wav").exists()]
missing += [i for i in test_ids if not (TEST_AUDIO / f"{i}.wav").exists()]
assert not missing, f"{len(missing)} audio files are missing, for example {missing[:3]}"
print(f"data folder : {DATA}")
print(f"audio       : {TRAIN_AUDIO}  |  {TEST_AUDIO}")
print(f"train {len(train_df)} rows, test {len(test_df)} rows, working folder {WORK}")
print(f"python      : {sys.executable}")

## 1. Labels and class priors

Only the verified recordings (`checked = 1`) are known to be correct, and the test labels are all verified. That makes the 620 verified recordings the best available stand-in for the test set. The competition page says every category has the same number of verified recordings; `train.csv` does not, and the notebook allows for that.

In [ ]:
# 1. Labels, the verified subset and the class priors ----------------------
classes = np.array(sorted(train_df["label"].unique()))
K = len(classes)
cls_idx = {c: i for i, c in enumerate(classes)}
y_all = train_df["label"].map(cls_idx).to_numpy().astype(int)
ver = train_df["checked"].to_numpy() == 1
ver_rows, unc_rows = np.flatnonzero(ver), np.flatnonzero(~ver)   # verified / unverified recordings
Yv = y_all[ver_rows]                                               # labels of the verified recordings

tab = pd.DataFrame({"verified": np.bincount(y_all[ver], minlength=K),
                    "unverified": np.bincount(y_all[~ver], minlength=K)}, index=classes)
tab["total"] = tab.verified + tab.unverified
print(f"{K} categories | verified {len(ver_rows)} | unverified {len(unc_rows)}")
print("verified clips per category:", tab.verified.value_counts().sort_index().to_dict())

# The test class mix is unknown. Two working assumptions are used only for weighting the
# cross-validation scores and for the prior correction: (a) the test set follows the class
# sizes in train.csv, (b) the test set is balanced across categories.
pi_target = tab.total.to_numpy(float) / tab.total.sum()
pi_ver    = tab.verified.to_numpy(float) / tab.verified.sum()
W_TARGET  = (pi_target / pi_ver)[Yv]          # verified clips re-weighted to look like assumption (a)
W_UNIFORM = ((1.0 / K) / pi_ver)[Yv]          # ... and like assumption (b)

## 2. Feature views

Each recording becomes up to three feature vectors. Every view is standardised using the training recordings only. If a view cannot be computed (for example, a model cannot be downloaded), the notebook says so and carries on with the others.

In [ ]:
# 2a. Feature view: librosa statistics (the same features your teammates used) -----
def load_or_compute(tag, make):
    """(train, test) feature matrices for one view, read from WORK when the cache matches this dataset."""
    f = WORK / f"{tag}_v1.npz"
    if f.exists():
        z = np.load(f, allow_pickle=True)
        if list(z["train_ids"]) == train_ids and list(z["test_ids"]) == test_ids:
            print(f"[{tag}] loaded from cache {f.name}")
            return z["Xtr"], z["Xte"]
    t0 = time.time()
    Xtr, Xte = make()
    np.savez_compressed(f, Xtr=Xtr, Xte=Xte, train_ids=np.array(train_ids), test_ids=np.array(test_ids))
    print(f"[{tag}] computed {Xtr.shape} and {Xte.shape} in {time.time() - t0:.0f}s")
    return Xtr, Xte

SR, N_FFT, HOP = 22050, 2048, 512

def handcrafted(path):
    try:
        y, sr = sf.read(str(path), dtype="float32")
        if y.ndim > 1:
            y = y.mean(axis=1)
        dur = len(y) / sr
        if len(y) < N_FFT:
            y = np.pad(y, (0, N_FFT - len(y)))
        S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)); P2 = S ** 2
        rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)
        zcr = librosa.feature.zero_crossing_rate(y=y, frame_length=N_FFT, hop_length=HOP)
        mel128 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=128)
        mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel128), n_mfcc=20)
        chroma = librosa.feature.chroma_stft(S=P2, sr=sr)
        cent = librosa.feature.spectral_centroid(S=S, sr=sr)
        bw = librosa.feature.spectral_bandwidth(S=S, sr=sr)
        roll = librosa.feature.spectral_rolloff(S=S, sr=sr)
        con = librosa.feature.spectral_contrast(S=S, sr=sr)
        ms = lambda M: np.concatenate([M.mean(axis=1), M.std(axis=1)])
        base = np.concatenate([ms(mfcc), ms(chroma), ms(cent), ms(bw), ms(roll), ms(zcr), ms(rms), [dur]])
        contrast = ms(con)
        r1 = rms[0]; T = S.shape[1]
        act = r1 >= max(np.percentile(r1, 30), 0.02 * r1.max())
        if act.sum() < 3:
            act = np.ones(T, bool)
        lm = librosa.power_to_db(librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=40))
        lma, mfa = lm[:, act], mfcc[:, act]
        dm = librosa.feature.delta(mfcc, width=9 if T >= 9 else 3, mode="nearest")
        flat = librosa.feature.spectral_flatness(S=S)[0]
        env = librosa.onset.onset_strength(S=lm, sr=sr, hop_length=HOP)
        n_on = len(librosa.onset.onset_detect(onset_envelope=env, sr=sr, hop_length=HOP))
        lr_ = np.log(r1 + 1e-8)
        rich = np.concatenate([
            lma.mean(1), lma.std(1), np.percentile(lma, 90, axis=1), lm.max(1),
            mfa.mean(1), mfa.std(1), dm.std(1),
            [flat.mean(), flat.std(), env.mean(), env.std()],
            np.percentile(lr_, [10, 50, 90]),
            [act.mean(), np.log(dur + 1e-3), n_on / max(dur, 1e-3)]])
        return np.concatenate([base, contrast, rich]).astype(np.float32)   # 319 numbers per clip
    except Exception:
        return None

def extract_handcrafted(ids, folder):
    rows = [handcrafted(folder / f"{i}.wav") for i in ids]
    if all(r is None for r in rows):
        raise RuntimeError(f"handcrafted features failed for every file in {folder}")
    dim = next(r.shape[0] for r in rows if r is not None)
    print(f"  {folder.parent.name}/{folder.name}: failed files = {sum(r is None for r in rows)}")
    return np.vstack([r if r is not None else np.full(dim, np.nan, np.float32) for r in rows])

VIEWS = {}
if USE_HAND:
    try:
        VIEWS["hand"] = load_or_compute("hand", lambda: (extract_handcrafted(train_ids, TRAIN_AUDIO),
                                                          extract_handcrafted(test_ids, TEST_AUDIO)))
    except Exception:
        import traceback
        print("hand view disabled. Full error:")
        traceback.print_exc()

In [ ]:
# 2b. Feature view: AudioSet-pretrained Audio Spectrogram Transformer (AST) ------
AST_ID  = "MIT/ast-finetuned-audioset-10-10-0.4593"
AST_WIN = 163840                     # 10.24 s at 16 kHz, the length AST was trained on

def make_ast():
    import torch
    # check for a GPU BEFORE importing transformers, so a CPU-only computer skips AST cleanly
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if dev == "cpu" and not AST_ON_CPU:
        raise RuntimeError("no GPU found, AST skipped (Kaggle: Settings > Accelerator > GPU)")
    from transformers import ASTFeatureExtractor, ASTForAudioClassification
    print("  AST running on", dev, "(the first call downloads about 350 MB of weights)")
    fe = ASTFeatureExtractor.from_pretrained(AST_ID)
    net = ASTForAudioClassification.from_pretrained(AST_ID).to(dev).eval()

    def load16(path):
        y, _ = librosa.load(str(path), sr=16000, mono=True)
        return np.pad(y, (0, max(0, 16000 - len(y)))).astype(np.float32)     # at least one second

    def windows(y):
        W = [y[s:s + AST_WIN] for s in range(0, len(y), AST_WIN)]
        return W[:-1] if len(W) > 1 and len(W[-1]) < 16000 else W           # drop a tiny tail

    @torch.inference_mode()
    def clip_feature(wins):
        x = fe(wins, sampling_rate=16000, return_tensors="pt")["input_values"].to(dev)
        ctx = torch.autocast("cuda", dtype=torch.float16) if dev == "cuda" else contextlib.nullcontext()
        with ctx:
            pooled = net.audio_spectrogram_transformer(x).pooler_output    # (windows, 768)
            logits = net.classifier(pooled)                                 # (windows, 527) AudioSet scores
        pooled, logits = pooled.float().cpu().numpy(), logits.float().cpu().numpy()
        return np.concatenate([pooled.mean(0), logits.mean(0), logits.max(0)])   # 768 + 527 + 527

    def run(paths, chunk=64):
        out = []
        with ThreadPoolExecutor(max_workers=4) as ex:
            for s in range(0, len(paths), chunk):
                for y in ex.map(load16, paths[s:s + chunk]):
                    out.append(clip_feature(windows(y)))
        return np.vstack(out).astype(np.float32)

    return (run([TRAIN_AUDIO / f"{i}.wav" for i in train_ids]),
            run([TEST_AUDIO / f"{i}.wav" for i in test_ids]))

if USE_AST:
    try:
        VIEWS["ast"] = load_or_compute("ast", make_ast)
    except Exception as e:
        print("ast view disabled:", repr(e)[:300])

In [ ]:
# 2c. Feature view: PANNs CNN14 embeddings --------------------------------------
def make_panns():
    import torch
    if importlib.util.find_spec("panns_inference") is None:
        os.system(f'"{sys.executable}" -m pip install -q panns-inference')
    from panns_inference import AudioTagging
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("  PANNs running on", dev, "(the first call downloads about 320 MB of weights)")
    at = AudioTagging(checkpoint_path=None, device=dev)

    def load32(path):
        x, _ = librosa.load(str(path), sr=32000, mono=True)
        return np.pad(x, (0, max(0, 32000 - len(x)))).astype(np.float32)

    def clip_feature(x):
        clipwise, emb = at.inference(x[None, :])        # (1, 527) AudioSet probabilities, (1, 2048) embedding
        return np.concatenate([emb[0], clipwise[0]])

    def run(paths, chunk=64):
        out = []
        with ThreadPoolExecutor(max_workers=4) as ex:
            for s in range(0, len(paths), chunk):
                for x in ex.map(load32, paths[s:s + chunk]):
                    out.append(clip_feature(x))
        return np.vstack(out).astype(np.float32)

    return (run([TRAIN_AUDIO / f"{i}.wav" for i in train_ids]),
            run([TEST_AUDIO / f"{i}.wav" for i in test_ids]))

if USE_PANNS:
    try:
        VIEWS["panns"] = load_or_compute("panns", make_panns)
    except Exception:
        import traceback
        print("panns view disabled. Full error:")
        traceback.print_exc()

In [ ]:
# 2d. Clean and standardise every view (the scaler is fitted on the training recordings) ---
X = {}                                   # view name -> (standardised train matrix, standardised test matrix)
for name, (Xtr, Xte) in VIEWS.items():
    Xtr, Xte = np.asarray(Xtr, np.float64), np.asarray(Xte, np.float64)
    med = np.nanmedian(Xtr, axis=0)
    Xtr = np.where(np.isnan(Xtr), med, Xtr)
    Xte = np.where(np.isnan(Xte), med, Xte)
    sc = StandardScaler().fit(Xtr)
    X[name] = (sc.transform(Xtr), sc.transform(Xte))
    print(f"view {name:6s}: {Xtr.shape[1]} dimensions")
assert X, "no feature view is available: run cells 2a to 2c in order and read the messages they print"
VIEW_NAMES = list(X)

## 3. Classifier, noisy labels and cross-validation

Five-fold cross-validation runs on the 620 verified recordings. Inside each fold, verified-only models judge the unverified recordings, and the three settings differ in how much of that judgement they use:

- `verified_only`: ignore the unverified recordings.
- `agree`: add an unverified recording only when the verified-only models rank its given label first.
- `soft`: add every unverified recording, weighted by how plausible its label looks.

Each setting is tried with two regularisation strengths. Scores are reported under two stated assumptions about the test set: its class mix follows the class sizes in `train.csv`, or it is balanced. The setting with the best average of the two is chosen.

In [ ]:
# 3. Classifier, noise-aware weighting and cross-validation on the verified clips -----
C_GRID = [0.03, 0.3]                     # regularisation strengths compared by cross-validation
CONFIGS = ["verified_only", "agree", "soft"]

def lr(Xm, y, w, C):
    return LogisticRegression(C=C, max_iter=300, tol=1e-3).fit(Xm, y, sample_weight=w)

def proba(model, Xm):
    P = np.zeros((Xm.shape[0], K))
    P[:, model.classes_] = model.predict_proba(Xm)
    return P

def balance(w, y):
    """Give every category the same total weight, then scale so the mean weight is 1."""
    tot = np.bincount(y, weights=w, minlength=K)
    w = w / np.maximum(tot, 1e-12)[y]
    return w / w.mean()

def unverified_weights(config, P_teacher, y_u):
    """How much to trust each unverified label, judged by the verified-only models."""
    if config == "agree":                                    # keep the labels the teacher ranks first
        return (P_teacher.argmax(1) == y_u).astype(float)
    if config == "soft":                                     # weight each label by how plausible it looks
        return P_teacher[np.arange(len(y_u)), y_u]
    raise ValueError(config)

def teacher_probs(rows_v, y_v):
    """Verified-only models, one per view, applied to every unverified clip, then averaged."""
    return np.mean([proba(lr(X[v][0][rows_v], y_v, balance(np.ones(len(rows_v)), y_v), C_LR), X[v][0][unc_rows])
                    for v in VIEW_NAMES], axis=0)

def training_set(config, P_teacher, rows_v, y_v):
    """Rows, labels and weights used by one configuration."""
    if config == "verified_only":
        return rows_v, y_v, balance(np.ones(len(rows_v)), y_v)
    wu = unverified_weights(config, P_teacher, y_all[unc_rows])
    keep = wu > 0
    rows = np.r_[rows_v, unc_rows[keep]]
    y = y_all[rows]
    return rows, y, balance(np.r_[np.ones(len(rows_v)), wu[keep]], y)

def run_fold(tr, va):
    """tr, va: positions inside the verified clips. Returns out-of-fold probabilities per (configuration, C, view)."""
    rows_v, rows_va = ver_rows[tr], ver_rows[va]
    y_v = y_all[rows_v]
    P_teacher = teacher_probs(rows_v, y_v)
    out = {}
    for c in CONFIGS:
        rows, y, w = training_set(c, P_teacher, rows_v, y_v)
        for C in C_GRID:
            out[(c, C)] = {v: proba(lr(X[v][0][rows], y, w, C), X[v][0][rows_va]) for v in VIEW_NAMES}
    return out

KEYS = [(c, C) for c in CONFIGS for C in C_GRID]
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=CV_SEED).split(np.zeros(len(Yv)), Yv))
t0 = time.time()
fold_out = []
for k, (tr, va) in enumerate(folds, 1):
    fold_out.append(run_fold(tr, va))
    print(f"  fold {k}/{N_FOLDS} finished after {time.time() - t0:.0f}s")

def oof(key, view):
    P = np.zeros((len(Yv), K))
    for (tr, va), out in zip(folds, fold_out):
        P[va] = out[key][view]
    return P

In [ ]:
# 3b. Scores under the two working assumptions, with temperature and prior correction ----
def map3_each(y, P):
    top = np.argsort(-P, axis=1)[:, :3]
    return ((top == y[:, None]) / np.array([1.0, 2.0, 3.0])).sum(1)

def sharpen(P, T):
    Q = np.clip(P, 1e-12, None) ** T
    return Q / Q.sum(1, keepdims=True)

def prior_adjust(P, gamma):
    Q = P * (pi_target ** gamma)[None, :]
    return Q / Q.sum(1, keepdims=True)

def nll(P, y, w):
    return -np.average(np.log(P[np.arange(len(y)), y] + 1e-12), weights=w)

def score(P_raw):
    T = min(np.linspace(0.2, 10.0, 99), key=lambda t: nll(sharpen(P_raw, t), Yv, W_TARGET))
    s = map3_each(Yv, prior_adjust(sharpen(P_raw, T), GAMMA))
    return dict(temperature=round(float(T), 2),
                plain_cv=s.mean(),
                test_like_cv=np.average(s, weights=W_TARGET),
                balanced_cv=np.average(s, weights=W_UNIFORM))

rows = []
for key in KEYS:
    r = score(np.mean([oof(key, v) for v in VIEW_NAMES], axis=0))     # average the views
    r["config"] = f"{key[0]}, C={key[1]}"
    r["key"] = key
    r["selection"] = 0.5 * (r["test_like_cv"] + r["balanced_cv"])
    rows.append(r)
RESULTS = pd.DataFrame(rows).set_index("config")
print("\nCross-validated MAP@3 on the 620 verified clips (five folds):")
print(RESULTS.drop(columns="key").round(4).to_string())
BEST = RESULTS["selection"].idxmax()
BEST_KEY = RESULTS.loc[BEST, "key"]
print("\nchosen setting:", BEST)
for v in VIEW_NAMES:
    print(f"  {v:6s} alone: plain CV MAP@3 = {map3_each(Yv, oof(BEST_KEY, v)).mean():.4f}")

## 4. Final model and submission

The chosen setting is retrained on all verified recordings and the selected unverified ones. The views are averaged, then calibrated and prior-corrected. `submission.csv` is the main file. The two files in `variants/` differ only in how strongly the prior correction is applied, in case you want to compare them on the leaderboard.

In [ ]:
# 4. Final model: all verified clips plus the chosen unverified clips, then the test set ---
def final_model(key):
    config, C = key
    rows_v, y_v = ver_rows, y_all[ver_rows]
    P_teacher = teacher_probs(rows_v, y_v)
    rows, y, w = training_set(config, P_teacher, rows_v, y_v)
    P_test = [proba(lr(X[v][0][rows], y, w, C), X[v][1]) for v in VIEW_NAMES]
    return np.mean(P_test, axis=0), len(rows) - len(rows_v)

P_raw_test, n_extra = final_model(BEST_KEY)
T_BEST = float(RESULTS.loc[BEST, "temperature"])
P_cal = sharpen(P_raw_test, T_BEST)
P_main = prior_adjust(P_cal, GAMMA)

def write_submission(P, path):
    top = np.argsort(-P, axis=1)[:, :3]
    sub = pd.DataFrame({"id": test_ids, "label": [" ".join(classes[t]) for t in top]})
    sub = sub.set_index("id").loc[sample_df["id"]].reset_index()    # same rows and order as sample_submission.csv
    assert len(sub) == len(sample_df) and all(len(set(s.split())) == 3 for s in sub["label"])
    sub.to_csv(path, index=False)
    return sub

VAR_DIR = WORK / "variants"
VAR_DIR.mkdir(exist_ok=True)
main = write_submission(P_main, WORK / "submission.csv")
write_submission(prior_adjust(P_cal, 0.0), VAR_DIR / "submission_no_prior.csv")
write_submission(prior_adjust(P_cal, 1.0), VAR_DIR / "submission_full_prior.csv")

share = np.bincount(P_main.argmax(1), minlength=K) / len(P_main)
print(f"submission written: {WORK / 'submission.csv'}  ({len(main)} rows)")
print(f"unverified clips used in the final model: {n_extra} of {len(unc_rows)}")
print("correlation of the predicted test mix with train class sizes: %.2f" % np.corrcoef(share, pi_target)[0, 1])
print("variant files:", sorted(p.name for p in VAR_DIR.glob("*.csv")))
print(main.head(3).to_string(index=False))

## 5. Submitting

1. Upload `submission.csv` with **Submit Prediction** on the competition page. Add a short description that names the setting, for example "AST + PANNs + librosa, soft, C=0.03".
2. Compare the public score with the cross-validated score in section 3. If the public score is far lower, check the messages in section 2 first to see which views loaded.
3. Your team has three submissions a day in total, so agree on which file to send before you upload.